### Pyspark


In [0]:
df =spark.read.format("csv")\
    .option("inferSchema", "true")\
    .option("header", True)\
    .load("/Volumes/praveen/bronze/bronze_volume/customers/")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
display(df)

In [0]:
df.printSchema()

In [0]:
df = df.withColumn("name",upper(col("name")))

In [0]:
display(df)

In [0]:
df = df.withColumn("domain",split(col("email"), "@")[1])
display(df)

In [0]:
display(df.groupBy("domain").agg(countDistinct(col("customer_id"))).alias("Total_customers"))


In [0]:
display(
    df.groupBy("domain")
    .agg(countDistinct(col("customer_id")))
    .withColumnRenamed('count(DISTINCT customer_id)', 'Total_customers')
    .sort(col("Total_customers").desc())
)

Databricks visualization. Run in Databricks to view.

In [0]:
df = df.withColumn("processDate",current_timestamp())
display(df)

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("praveen.silver.customers_err"):
    
    dlt_obj = DeltaTable.forName(spark, "praveen.silver.customers_err")
    dlt_obj.alias("trg").merge(df.alias("src"),"trg.customer_id == src.customer_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
    

else:
    df.write.format("delta")\
        .mode("append")\
            .saveAsTable("praveen.silver.customers_err")